File Name: pol_RBG_image_saocom.ipynb

Authors:
    Juan M. Lopez-Sanchez  
    University of Alicante

Date: July 2025

Description:
------------
This script generates a false colour RGB image for a dual-pol (VV-VH) SAOCOM image in POL format.

The input image must have been previously processed and should be available 
in POL format (either as COV or GCOV).

Dependencies:
-------------
- numpy
- os
- pathlib
- PIL
- rosel.io
- rosel.computation
- rosel.shiroma

Usage:
------
To use this script, define the following parameters:
    
    - common_folder: Common path to all products, e.g. '/media/s3-ard-rose-l/PHASE_1/DEMONSTRATOR'
    - site: 'Naples', 'Salamanca', or 'BlackForest'
    - orbit: 'ASC' or 'DSC'
    - beam: 'S3' or 'S4' 
    - multilook_option: 'SHIROMA' (without multi-look), 'MULTI-LOOK', or 'BOXCAR'
    - multilook_size: e.g. '9x5', '11x7', 15x9', etc. Not used if multilook_option is 'SHIROMA' 
    - product: 'COV' or 'GCOV'
    - acquisition_date: time stamp of the original product, from with the date is used, e.g. '2021-02-28T17:09:09.724349' # Available for DSC/S4 Naples


Package dependencies

In [1]:
import numpy as np
import rosel.io
import rosel.computation
import rosel.shiroma


import os
import pathlib

from PIL import Image

Auxiliary functions

In [2]:

def bytescl( data, lim, mask = None ):
    
    data_val = np.clip( (data - lim[0]) / (lim[1] - lim[0]), 0, 1)
                
    data_byte = np.zeros_like( data_val, dtype=np.uint8)
    
    if mask is None:
           data_byte[:] = 255 * data_val[:]
    else:
           data_byte[mask] = 255 * data_val[mask]
    
    return data_byte

def show_rgb_C2( C11, C22, C12 ):
    
    # Show an RGB image from C11 and C22.
    # C12 is not used.
    
    Nrows, Ncols = C11.shape
    
    # dB scale
    
    C11_dB = 10*np.log10( C11 + 1e-12)
    C22_dB = 10*np.log10( 2*C22 + 1e-12)
    CH_dB =  10*np.log10( C11 / (C22 + 1e-12) + 1e-12)
    
    # Dynamic range
    
    mindB = -25
    maxdB = 5
    mindif = 0
    maxdif = 20
    
    # Create the RGB image
    
    RGB_image = np.zeros([Nrows, Ncols, 3], dtype=np.uint8)
    
    RGB_image[:,:,0] = bytescl( C11_dB, [mindB, maxdB]) # R
    RGB_image[:,:,1] = bytescl( C22_dB, [mindB, maxdB])  # G
    RGB_image[:,:,2] = bytescl( CH_dB, [mindif, maxdif]) # B
       
    Image.fromarray( RGB_image ).show()

User-defined input parameters 

In [3]:
########################
# PARAMETERS
########################

common_folder = '/media/s3-ard-rose-l/PHASE_1/DEMONSTRATOR'

site = 'Naples' # Naples, Salamanca, BlackForest
orbit = 'DSC'   # ASC, DSC
beam = 'S4'      # S3, S4

multilook_option = 'MULTI-LOOK' # 'SHIROMA' (wthiout multi-look), 'MULTI-LOOK' or 'BOXCAR'
multilook_size = '11x7' # e.g. '9x5', '11x7', 15x9'. Not used if multilook_option is 'SHIROMA'

product = 'GCOV' # 'COV' or 'GCOV'

# Date selection
# Example:
acquisition_date = '2021-02-28T17:09:09.724349' # Available for DSC/S4 Naples

Definition of input folder with the POL product to be used (by means of the user-defined parameters)

In [4]:
# Date format (as subfolder of POL products)

date_txt = acquisition_date[:10] # only the string with the date

# Form path to input POL product

polsar_folder = os.path.join( common_folder, 'POL', site, orbit, beam, multilook_option)

if multilook_option != 'SHIROMA':
    polsar_folder = os.path.join( polsar_folder, multilook_size)

input_folder = os.path.join( polsar_folder, date_txt )

PROCESSING

In [5]:

# Read the 3 layers of a POL product

C11 = rosel.io.read( os.path.join( input_folder, product, 'C11'), consolidate_metadata = False ).v
C22 = rosel.io.read( os.path.join( input_folder, product, 'C22'), consolidate_metadata = False ).v
C12 = rosel.io.read( os.path.join( input_folder, product, 'C12'), consolidate_metadata = False ).v

In [6]:

# RGB representation (using only the intensities of the two channels)

show_rgb_C2( C11, C22, C12 )




